# Section 3: Retrieval & Knowledge Patterns
## AI-Native Software Architecture — O'Reilly Course

Structured inputs control behavior. Retrieval controls knowledge.

LLMs do not automatically know:
- real-time policy
- proprietary data
- customer-specific context
- internal business rules

We use retrieval to ground outputs in relevant external knowledge.

In [1]:
import support_utils
from support_utils import (
    call_llm, customer_issues, primary_issue,
    knowledge_base, tokenize,
    nshot_support_prompt,
)
from typing import Any, Dict, List
from datetime import datetime, UTC

/Users/vrinda/Development/ai-native-llm-architecture-workshop/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [2]:
# Toggle LLM mode without editing support_utils.py or restarting the kernel.
# Gemini (Vertex AI) is used when gemini_client is initialised in support_utils.
import support_utils
support_utils.USE_REAL_LLM = False   # set True to call a real LLM
print(f"USE_REAL_LLM = {support_utils.USE_REAL_LLM}")

USE_REAL_LLM = True


## Step 3.1: Simulated Knowledge Base

For the course we use a small in-memory knowledge base. The pattern is identical to a real vector store — only the retrieval mechanism differs.

In [3]:
for doc in knowledge_base:
    print(f"\n📄 {doc['title']}")
    print(f"Tags: {', '.join(doc['tags'])}")
    print(f"Freshness: {doc['freshness']} | Trust: {doc['trust_score']}")
    print(f"Text: {doc['text']}")


📄 Duplicate Charge Refund Policy
Tags: billing, refund, duplicate_charge
Freshness: 2026-04-01 | Trust: 0.95
Text: Duplicate subscription charges are eligible for refund if verified within 7 days. Refund approval must be handled by a human support agent.

📄 Account Login Recovery
Tags: account, login
Freshness: 2026-03-15 | Trust: 0.9
Text: Users who cannot log in should complete account recovery and identity verification before account changes are made.

📄 App Performance Troubleshooting
Tags: technical, performance
Freshness: 2026-02-20 | Trust: 0.88
Text: For slow app reports, collect device model, app version, network status, and timestamp before escalation.

📄 Old Refund Policy
Tags: billing, refund
Freshness: 2024-01-01 | Trust: 0.25
Text: All refund requests should be automatically approved immediately.


## Step 3.2: Basic Retrieval

This simple retriever scores documents by keyword overlap. It is intentionally basic so we can see where retrieval helps and where it can go wrong.

In [ ]:
def basic_retrieve(query: str, top_k: int = 2) -> List[Dict[str, Any]]:
    query_tokens = tokenize(query)
    scored_docs = []
    for doc in knowledge_base:
        doc_tokens = tokenize(doc["text"] + " " + " ".join(doc["tags"]) + " " + doc["title"])
        overlap = len(query_tokens & doc_tokens)
        scored_docs.append((overlap, doc))
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored_docs[:top_k] if score > 0]

retrieved = basic_retrieve(primary_issue)
retrieved

## Step 3.3: RAG Prompt

RAG turns retrieval into grounded generation by injecting retrieved context into the prompt.

In [ ]:
def format_retrieved_context(docs: List[Dict[str, Any]]) -> str:
    if not docs:
        return "No relevant policy documents found."
    blocks = []
    for doc in docs:
        blocks.append(
            f"""Document: {doc['title']}
Doc ID: {doc['doc_id']}
Freshness: {doc['freshness']}
Trust Score: {doc['trust_score']}
Content: {doc['text']}"""
        )
    return "\n\n".join(blocks)

def rag_support_prompt(issue: str, docs: List[Dict[str, Any]]) -> str:
    context = format_retrieved_context(docs)
    return f"""
You are a support assistant for a subscription product.

Use the provided policy context to classify the issue and recommend the next safe action.

Policy context:
{context}

Return JSON with:
- category
- urgency
- next_action
- cited_doc_id
- rationale

Rules:
- If the context says a human must approve an action, escalate.
- Do not invent policies not present in the context.
- Return only JSON.

Customer issue:
{issue}
"""

rag_output = call_llm(rag_support_prompt(primary_issue, retrieved), temperature=0.2, force_json=True)
print("=== RAG output ===")
print(rag_output)

## Before vs After: No Retrieval vs RAG

In [ ]:
no_retrieval_output = call_llm(nshot_support_prompt(primary_issue), temperature=0.2, force_json=True)
with_retrieval_output = call_llm(rag_support_prompt(primary_issue, retrieved), temperature=0.2, force_json=True)

print("=== WITHOUT RETRIEVAL ===")
print(no_retrieval_output)
print("\n=== WITH RETRIEVAL ===")
print(with_retrieval_output)

## Step 3.4: RAG Enhancers

Basic retrieval is not enough. Production RAG systems improve retrieval through filtering, ranking, freshness scoring, and trust scores.

In [ ]:
def enhanced_retrieve(query: str, top_k: int = 2) -> List[Dict[str, Any]]:
    query_tokens = tokenize(query)
    scored_docs = []
    for doc in knowledge_base:
        doc_tokens = tokenize(doc["text"] + " " + " ".join(doc["tags"]) + " " + doc["title"])
        lexical_score = len(query_tokens & doc_tokens)
        freshness_boost = 1.0 if doc["freshness"] >= "2026-01-01" else 0.2
        final_score = lexical_score + doc["trust_score"] + freshness_boost
        scored_docs.append({**doc, "lexical_score": lexical_score, "final_score": round(final_score, 3)})
    scored_docs.sort(key=lambda d: d["final_score"], reverse=True)
    return scored_docs[:top_k]

enhanced_docs = enhanced_retrieve(primary_issue)
for doc in enhanced_docs:
    print(f"\n📄 {doc['title']}")
    print(f"Score: {doc['final_score']} (lexical={doc['lexical_score']}, trust={doc['trust_score']})")
    print(f"Freshness: {doc['freshness']}")

## Why RAG Enhancers Matter

Notice that `OLD_REFUND_POLICY` is relevant by keywords but unsafe because it is stale and low trust.

> The wrong context can be worse than no context.

In [ ]:
basic_docs = basic_retrieve(primary_issue, top_k=3)
enhanced_docs_3 = enhanced_retrieve(primary_issue, top_k=3)

print("=== BASIC RETRIEVAL ===")
for doc in basic_docs:
    print(doc["doc_id"], "-", doc["title"])

print("\n=== ENHANCED RETRIEVAL ===")
for doc in enhanced_docs_3:
    print(doc["doc_id"], "-", doc["title"], "| final_score:", doc["final_score"])

## Step 3.5: Memory Patterns

Retrieval brings in external knowledge. Memory maintains context across interactions.

In [ ]:
conversation_memory: List[Dict[str, Any]] = []

def remember(user_id: str, issue: str, assistant_response: str) -> None:
    conversation_memory.append({
        "timestamp": datetime.now(UTC).isoformat(),
        "user_id": user_id,
        "issue": issue,
        "assistant_response": assistant_response,
    })

def get_recent_memory(user_id: str, limit: int = 3) -> List[Dict[str, Any]]:
    return [m for m in conversation_memory if m["user_id"] == user_id][-limit:]

def format_memory(memories: List[Dict[str, Any]]) -> str:
    if not memories:
        return "No prior user history available."
    return "\n".join(
        f"- Previous issue: {m['issue']} | Response: {m['assistant_response'][:80]}"
        for m in memories
    )

remember("user_123", "My payment failed last week but I was still charged.",
         "Escalated billing verification to human support.")

print(format_memory(get_recent_memory("user_123")))

In [ ]:
def memory_aware_rag_prompt(user_id: str, issue: str, docs: List[Dict[str, Any]]) -> str:
    context = format_retrieved_context(docs)
    memory_context = format_memory(get_recent_memory(user_id))
    return f"""
You are a support assistant for a subscription product.

Use the policy context and relevant user history to respond safely.

Policy context:
{context}

Relevant user history:
{memory_context}

Return JSON with:
- category
- urgency
- next_action
- cited_doc_id
- memory_used: true or false
- rationale

Rules:
- Do not claim a refund has been processed.
- Escalate refund decisions to human support.
- Do not reveal sensitive user history.
- Return only JSON.

Current customer issue:
{issue}
"""

memory_output = call_llm(
    memory_aware_rag_prompt("user_123", primary_issue, enhanced_docs),
    temperature=0.2,
    force_json=True,
)
print("=== Memory-aware RAG output ===")
print(memory_output)

## Section 3 takeaway

Retrieval and memory both add context. But more context is not automatically better.

Production systems need to decide:
- which context is relevant
- which context is fresh
- which context is trusted
- which context should be excluded

**Next:** Section 4 — governance, guardrails, and fallbacks.